In [10]:
import pandas as pd

In [11]:
path_sygen = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/sygen_2025_clean.csv"
path_finess = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/finess_structures_clean.csv"

output_path = "s3://mangane/Projet_PDS_2026/processed/finess_sygen_final.csv"

In [12]:
df_sygen = pd.read_csv(
    path_sygen,
    dtype={
        "FI": str,
        "FI_EJ": str
    }
)

df_finess = pd.read_csv(
    path_finess,
    dtype={
        "FI": str,
        "FI_EJ": str,
        "CODGEO": str,
        "CODE_POSTAL": str,
        "CATEGORIE": str,
        "SIRET": str
    }
)

print("SYGEN :", df_sygen.shape)
print("FINESS :", df_finess.shape)

SYGEN : (3988, 29)
FINESS : (174840, 15)


In [13]:
print("Doublons FI dans SYGEN :", df_sygen["FI"].duplicated().sum())
print("Doublons FI dans FINESS :", df_finess["FI"].duplicated().sum())

print("FI manquants dans SYGEN :", df_sygen["FI"].isna().sum())
print("FI manquants dans FINESS :", df_finess["FI"].isna().sum())

Doublons FI dans SYGEN : 0
Doublons FI dans FINESS : 0
FI manquants dans SYGEN : 0
FI manquants dans FINESS : 0


In [14]:
colonnes_finess = [
    "FI",
    "FI_EJ",
    "NOM",
    "CATEGORIE",
    "ETAT",
    "DATE_OUVERTURE",
    "DATE_FERMETURE",
    "SIRET",
    "CODGEO",
    "CODE_POSTAL",
    "ADRESSE",
    "COMMUNE",
    "LONGITUDE",
    "LATITUDE",
    "SCORE_BAN"
]

df_fusion = df_sygen.merge(
    df_finess[colonnes_finess],
    on="FI",
    how="left",
    suffixes=("_SYGEN", "_FINESS"),
    indicator=True
)

print(df_fusion.shape)
print(df_fusion["_merge"].value_counts())

(3988, 44)
_merge
both          3940
left_only       48
right_only       0
Name: count, dtype: int64


In [15]:
taux_appariement = (
    (df_fusion["_merge"] == "both").mean() * 100
)

print("Taux d'appariement :", round(taux_appariement, 1), "%")
print("Etablissements non retrouvés :", (df_fusion["_merge"] == "left_only").sum())

Taux d'appariement : 98.8 %
Etablissements non retrouvés : 48


In [16]:
masque_ej = (
    df_fusion["FI_EJ_SYGEN"].notna()
    & df_fusion["FI_EJ_FINESS"].notna()
)

nb_ecarts_ej = (
    df_fusion.loc[masque_ej, "FI_EJ_SYGEN"]
    != df_fusion.loc[masque_ej, "FI_EJ_FINESS"]
).sum()

print("Ecarts FI_EJ SYGEN / FINESS :", nb_ecarts_ej)

Ecarts FI_EJ SYGEN / FINESS : 47


In [17]:
df_non_apparies = df_fusion.loc[
    df_fusion["_merge"] == "left_only",
    ["FI", "FI_EJ_SYGEN", "AN"]
].copy()

print(df_non_apparies.shape)
df_non_apparies.head(10)

(48, 3)


,FI,FI_EJ_SYGEN,AN
43,020000600,020000600,2025
199,060800174,060800174,2025
307,110786324,110786324,2025
340,130001696,130001696,2025
350,130010648,130010648,2025
459,130804032,130804032,2025
610,190000117,190000117,2025
670,220000210,220000210,2025
968,300000387,300000387,2025
970,300000759,300000759,2025


In [18]:
df_fusion["FI_EJ"] = df_fusion["FI_EJ_SYGEN"]

df_fusion = df_fusion.drop(
    columns=[
        "FI_EJ_SYGEN",
        "FI_EJ_FINESS",
        "_merge"
    ]
)

colonnes_debut = [
    "FI",
    "FI_EJ",
    "AN",
    "NOM",
    "CATEGORIE",
    "ETAT",
    "CODGEO",
    "CODE_POSTAL",
    "COMMUNE",
    "ADRESSE",
    "LONGITUDE",
    "LATITUDE"
]

autres_colonnes = [
    c for c in df_fusion.columns
    if c not in colonnes_debut
]

df_fusion = df_fusion[
    colonnes_debut + autres_colonnes
]

print(df_fusion.shape)
df_fusion.head()

(3988, 42)


,FI,FI_EJ,AN,NOM,CATEGORIE,ETAT,CODGEO,CODE_POSTAL,COMMUNE,ADRESSE,...,ETPSAL_TOT,EFFPNM_TOT_PNM,ETPPNM_TOT_PNM,NB_SCAN,NB_IRM,NB_MAMM,DATE_OUVERTURE,DATE_FERMETURE,SIRET,SCORE_BAN
0,010000024,010780054,2025,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,355,A,01451,01440,VIRIAT,900 ROUTE DE PARIS,...,181.07,1952.0,1811.92,2.0,3.0,0.0,1979-02-13,NaN,26010004500012,0.963911
1,010000032,010780062,2025,CENTRE HOSPITALIER BUGEY SUD,355,A,01034,01300,BELLEY,700 AVENUE DE NARVIK,...,27.69,438.0,380.05,1.0,0.0,1.0,1901-01-01,NaN,26010003700068,0.951978
2,010000065,010780096,2025,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,355,A,01427,01606,TREVOUX CEDEX,14 RUE DE L'HOPITAL,...,18.26,264.0,225.23,0.0,0.0,0.0,1901-01-01,NaN,26010028400017,0.956661
3,010000081,010780112,2025,CENTRE HOSPITALIER DU PAYS DE GEX,106,A,01173,01174,GEX CEDEX,160 RUE MARC PANISSOD,...,1.83,77.0,64.16,NaN,NaN,NaN,1901-01-01,NaN,26010010200011,0.961420
4,010000099,010780120,2025,CENTRE HOSPITALIER DE MEXIMIEUX,106,A,01244,01800,MEXIMIEUX,13 AVENUE DU DOCTEUR BOYER,...,2.59,66.0,42.99,NaN,NaN,NaN,1945-01-01,NaN,26010013600019,0.964063


In [19]:
print("Doublons FI :", df_fusion["FI"].duplicated().sum())
print("CODGEO manquants :", df_fusion["CODGEO"].isna().sum())

print("\nValeurs manquantes :")
print(
    df_fusion.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

Doublons FI : 0
CODGEO manquants : 48

Valeurs manquantes :
DATE_FERMETURE    3969
PLATOT_HAD        3708
SAMU              3530
CAP_HDJ_TOT       3442
PLA_OBS           3440
LIT_USLD          3416
AUTPED            3371
AUTGEN            3371
CAP_HC_TOT        3354
LIT_OBS           3328
PLA_CHI           3031
LIT_CHI           3028
NB_MAMM           2893
NB_SCAN           2893
NB_IRM            2893
dtype: int64


In [20]:
df_fusion.to_csv(
    output_path,
    index=False
)

print("Fichier sauvegardé :", output_path)

Fichier sauvegardé : s3://mangane/Projet_PDS_2026/processed/finess_sygen_final.csv
